In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
import json
import os
from datetime import datetime

In [2]:
def get_the_access(auth_url='https://auth.fractalite.com/auth/realms/master/protocol/openid-connect/token',
                   username="ml@fractalite.com",
                   scope="offline_access",
                   grant_type="password",
                   client_id="ml-beds",
                   client_secret="b33425cc-3230-48a7-8bfa-b2b779286b8c",
                   password="fractalite"):
    
    auth_payload = {
        "client_id": client_id,
        "client_secret": client_secret,
        "username": username,
        "password": password,
        "grant_type": grant_type,
        "scope": scope
    }

    token_response = requests.post(auth_url, data=auth_payload)

    if token_response.status_code == 200:
        token_data = token_response.json()
        access_token = token_data["access_token"]
        print("Token obtained successfully!")
        return access_token
    else:
        print(f"Error {token_response.status_code}: {token_response.text}")
        return None
    
def get_the_data_api(api_url='http://beds-api.fractalite.com/properties/147/rate-bases', 
                     access_token=None):
    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json"
    }
    try:
        response = requests.get(api_url, headers=headers)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.HTTPError as err:
        print(f"The are error in the request: {err}")
        return None
    
def get_data(api_url):
    access_token = get_the_access()
    data = get_the_data_api(access_token=access_token, api_url=api_url)
    return data

def append_to_json_file(data, file):
    if os.path.exists(file):
        return 
    
    with open(file, 'w', encoding='utf-8') as json_file:
        json.dump(data, json_file, indent=4, ensure_ascii=False)
    print(f"Data saved to {file}")

In [3]:
def meregd_allotments_rateBasses(allotments_path, rateBasses_path):
    """
    Based on the allotments and rateBasses data, "bases" and "code"
    """

    with open(allotments_path, 'r') as json_file:
        allotments = json.load(json_file)
    
    with open(rateBasses_path, 'r') as json_file:
        rateBasses = json.load(json_file)

    rateBasses_lookup = {item['code']: item for item in rateBasses}

    meregd_data = []

    for allotment in allotments:
        calendar = allotment.pop('calendar', None)
        merged_item = allotment.copy()
        bases_codes = allotment.get('bases', [])

        if bases_codes and bases_codes[0] in rateBasses_lookup:
            rate_base = rateBasses_lookup[bases_codes[0]]
            merged_item.update({
                "property": rate_base.get("property"),
                "booking_period": rate_base.get("booking_period"),
                "stay_period": rate_base.get("stay_period"),
                "sales_segments": rate_base.get("sales_segments"),
                "rate_code": rate_base.get("code"),
                "description": rate_base.get("description"),
                "commission": rate_base.get("commission"),
                "payOnArrival": rate_base.get("payOnArrival"),
                "currency": rate_base.get("currency"),
                "taxInc": rate_base.get("taxInc"),
            })
        
        if calendar:
            merged_item['calendar'] = calendar

        meregd_data.append(merged_item)

    return meregd_data

In [4]:
merged_data = meregd_allotments_rateBasses('../Data/MainData/allotmentsData.json', '../Data/MainData/rateBaseData.json')
append_to_json_file(merged_data, '../Data/UnitsTransformedData/AllotmentsRateBase.json')

Data saved to ../Data/UnitsTransformedData/AllotmentsRateBase.json


In [5]:
def flattenData(allotments_rateBasses_path):
    """
    Flatten the allotments and rateBasses data based on the calendar entry 
    """

    with open(allotments_rateBasses_path, 'r') as json_file:
        allotments_rateBasses = json.load(json_file)
    
    flattened_data = []

    for entry in allotments_rateBasses:
        calendar_entries = entry.pop('calendar', [])
        base_info = entry.copy()

        for calendar_entry in calendar_entries:
            flattened_entry = {**calendar_entry, **base_info}
            flattened_data.append(flattened_entry)
    
    return flattened_data

In [6]:
flattened_data = flattenData('../Data/UnitsTransformedData/AllotmentsRateBase.json')
append_to_json_file(flattened_data, '../Data/UnitsTransformedData/FlattenedAllotmentsRateBase.json')

Data saved to ../Data/UnitsTransformedData/FlattenedAllotmentsRateBase.json


In [7]:
def merged_product_rateBasses(product_path, rateBasses_path):
    """
    Merged the product and rateBasses data based on the "code"
    """

    with open(product_path, 'r') as json_file:
        products = json.load(json_file)
    
    with open(rateBasses_path, 'r') as json_file:
        rateBasses = json.load(json_file)
    
    rateBasses_lookup = {item['code']: item for item in rateBasses}
    merged_data = []

    for product in products:
        calendar = product.pop('calendar', None)
        merged_item = product.copy()
        bases_codes = product.get('rateBasis')

        if bases_codes and bases_codes in rateBasses_lookup:
            rate_base = rateBasses_lookup[bases_codes]
            merged_item.update({
                "property": rate_base.get("property"),
                "booking_period": rate_base.get("booking_period"),
                "stay_period": rate_base.get("stay_period"),
                "sales_segments": rate_base.get("sales_segments"),
                "rate_code": rate_base.get("code"),
                "description": rate_base.get("description"),
                "commission": rate_base.get("commission"),
                "payOnArrival": rate_base.get("payOnArrival"),
                "currency": rate_base.get("currency"),
                "taxInc": rate_base.get("taxInc"),
            })
        
        if calendar:
            merged_item['calendar'] = calendar
        
        merged_data.append(merged_item)
    
    return merged_data

In [8]:
merged_data = merged_product_rateBasses('../Data/MainData/productsData.json', '../Data/MainData/rateBaseData.json')
append_to_json_file(merged_data, '../Data/UnitsTransformedData/ProductRateBase.json')

Data saved to ../Data/UnitsTransformedData/ProductRateBase.json


In [9]:
flattened_data = flattenData('../Data/UnitsTransformedData/ProductRateBase.json')
append_to_json_file(flattened_data, '../Data/UnitsTransformedData/FlattenedProductRateBase.json')

Data saved to ../Data/UnitsTransformedData/FlattenedProductRateBase.json


In [10]:
def structure_date(data_path):
    """
    Structure the data as "date" is the main key of all of entry
    """
    with open(data_path, 'r') as json_file:
        data = json.load(json_file)
    
    structured_data = {}
    for entry in data:
        date = entry.get('date')
        unit = entry.get('unit')

        if date:
            if date not in structured_data:
                structured_data[date] = {}
            
            if unit not in structured_data[date]:
                structured_data[date][unit] = []
            
            structured_data[date][unit].append(entry)
    
    return structured_data

In [11]:
structured_data = structure_date('../Data/UnitsTransformedData/FlattenedAllotmentsRateBase.json')
append_to_json_file(structured_data, '../Data/UnitsTransformedData/StructuredAllotmentsRateBase.json')

Data saved to ../Data/UnitsTransformedData/StructuredAllotmentsRateBase.json


In [12]:
productStructedData = structure_date('../Data/UnitsTransformedData/FlattenedProductRateBase.json')
append_to_json_file(productStructedData, '../Data/UnitsTransformedData/StructuredProductRateBase.json')

Data saved to ../Data/UnitsTransformedData/StructuredProductRateBase.json


In [13]:
def combine_allotments_product(product_path, allotments_path, eur_to_mad_rate=10.42):
    """
    For given total allotments and product data, calculate the total price and total availability.
    Handles None values by converting them to 0.0
    """
    with open(product_path, 'r') as json_file:
        products_data = json.load(json_file)
    
    with open(allotments_path, 'r') as json_file:
        allotments_data = json.load(json_file)
    
    combined_data = {}
    all_dates = set(list(products_data.keys()) + list(allotments_data.keys()))

    for date in all_dates:
        combined_data[date] = {}
        
        units_prices = set(products_data.get(date, {}).keys())
        units_allotments = set(allotments_data.get(date, {}).keys())
        all_units = units_prices.union(units_allotments)
        
        for unit in all_units:
            unit_data = {
                "price_totals": {  
                    "total_price": 0.0,
                    "total_public": 0.0,
                    "currency": "MAD"
                },
                "allotment_totals": {
                    "total_available": 0,
                    "total_booked": 0
                },
                "products_entries": products_data.get(date, {}).get(unit, []),  
                "allotments_entries": allotments_data.get(date, {}).get(unit, [])
            }
            
            for entry in unit_data["products_entries"]:
                price = entry.get("price")
                currency = entry.get("currency")
                public = entry.get("public")

                if currency == "EUR":
                    price = float(price if price is not None else 0.0) * eur_to_mad_rate
                    public = float(public if public is not None else 0.0) * eur_to_mad_rate
                else:
                    price = float(price if price is not None else 0.0)
                    public = float(public if public is not None else 0.0)

                unit_data["price_totals"]["total_price"] += float(price if price is not None else 0.0)
                unit_data["price_totals"]["total_public"] += float(public if public is not None else 0.0)
            
            for entry in unit_data["allotments_entries"]:
                unit_data["allotment_totals"]["total_available"] += entry.get("available", 0)
                unit_data["allotment_totals"]["total_booked"] += entry.get("booked", 0)
            
            combined_data[date][unit] = unit_data
    
    return combined_data

In [14]:
combined_data = combine_allotments_product('../Data/UnitsTransformedData/StructuredProductRateBase.json', '../Data/UnitsTransformedData/StructuredAllotmentsRateBase.json')
append_to_json_file(combined_data, '../Data/UnitsTransformedData/CombinedData.json')

Data saved to ../Data/UnitsTransformedData/CombinedData.json


In [15]:
def filterDateRange(comined_data_path, start_date="2024-01-01", end_date="2025-12-31"):
    """
    Filtered data based on the given date range
    """
    with open(comined_data_path, 'r') as json_file:
        combined_data = json.load(json_file)
    
    filtered_data = {}
    start = datetime.strptime(start_date, "%Y-%m-%d")
    end = datetime.strptime(end_date, "%Y-%m-%d")

    for date in combined_data:
        current_date = datetime.strptime(date, "%Y-%m-%d")
        if start <= current_date <= end:
            filtered_data[date] = combined_data[date]
    
    return filtered_data

In [16]:

filtered_data = filterDateRange('../Data/UnitsTransformedData/CombinedData.json', start_date="2024-01-01", end_date="2025-12-31")
append_to_json_file(filtered_data, '../Data/UnitsTransformedData/CombinedData2024_2025.json')

Data saved to ../Data/UnitsTransformedData/CombinedData2024_2025.json


In [17]:
def calculate_taux(combined_data_path):
    """
    Calculate the taux d'occupation and taux de remplissage based on the combined dat
    """
    with open(combined_data_path, 'r') as json_file:
        combined_data = json.load(json_file)

    enriched_data = {}
    
    for date, units in combined_data.items():
        enriched_data[date] = {}
        
        for unit, data in units.items():
            unit_data = data.copy()

            total_available = unit_data["allotment_totals"]["total_available"]
            total_booked = unit_data["allotment_totals"]["total_booked"]

            total_available = float(total_available if total_available is not None else 0.0)
            total_booked = float(total_booked if total_booked is not None else 0.0)
            allotments_count = len(unit_data.get("allotments_entries", []))

            occupancy_rate = 0.0
            if total_available > 0:
                occupancy_rate = round((total_booked / total_available) * 100, 2)
            
            remaining = total_available - total_booked

            unit_data['allotment_totals'].update({
                "taux_occupation": occupancy_rate,
                "taux_remplissage": remaining,
                "allotments_count": allotments_count,
            })

            enriched_data[date][unit] = unit_data

    return enriched_data

In [18]:

data = calculate_taux('../Data/UnitsTransformedData/CombinedData2024_2025.json')
append_to_json_file(data, '../Data/UnitsTransformedData/DataFinalTransformed.json')

Data saved to ../Data/UnitsTransformedData/DataFinalTransformed.json
